In [ ]:
#| echo: false
import os

import retentioneering as rete
from IPython.display import HTML

# Rendered by Quarto, slides have no Python kernel behind them, so every widget
# is baked in as a self-contained static page (render_static) instead of a live
# anywidget, with its settings sidebar closed to leave the slide to the
# chart. Run in Jupyter, the widgets stay live.
if "QUARTO_DOCUMENT_PATH" in os.environ:
    from retentioneering.widgets._base import RetentioneeringWidget

    # A widget can ask for its own height on a slide: `widget.slide_height = …`
    # (the widget's layout follows its `height`, the iframe follows the page).
    SLIDE_WIDGET_HEIGHT = 470

    def _static_bundle(self, include=None, exclude=None, **kwargs):
        height = getattr(self, "slide_height", None)
        if height:
            self.height = height
        return {
            "text/html": self.render_static(
                height=height or SLIDE_WIDGET_HEIGHT, sidebar_open=False
            ).data
        }

    RetentioneeringWidget._repr_mimebundle_ = _static_bundle

## About us

### Vladimir Kukushkin

![](img/vladimir-kukushkin.jpg)

- 13 years in product analytics
- ex. Member of Techincal Staff @ Perplexity
- Principal AI Quality Analyst @ Finom
- Co-founder of Retentioneering

### Maxim Godzi

![](img/maxim-godzi.jpg)

- PhD in XXX
- NN years in product analytics
- ex. Principal Data Science Manager @ Microsoft
- Co-founder of Retentioneering

# Part 1: Motivating example

## How does a team adopt a task tracker?

**Taskboard**, a task tracker for teams: 20,000 users over eight weeks, 144,332 visits, 742,922 events of 31 types.

In [ ]:
#| output: false
from taskboard import load_taskboard

taskboard = load_taskboard()

| Getting started | Working alone | Working together | Power features | Planning | Paying |
|---|---|---|---|---|---|
| `signup` | `open_board` | `invite_member` | `view_report` | `view_timeline` | `view_pricing` |
| `tour_step` | `create_task` | `view_invite_status` | `create_automation` | `set_due_date` | `select_plan` |
| `template_view` | `complete_task` | `resend_invite` | `connect_integration` | `add_dependency` | `start_checkout` |
| `create_project` | `edit_task` | `accept_invite` | `export` |  | `enter_payment` |
|  | `view_notifications` | `teammate_joined` |  |  | `payment_error` |
|  |  | `comment` |  |  | `support_ticket` |
|  |  | `mention` |  |  | `upgrade` |
|  |  | `assign_task` |  |  |  |

**Happy path:** `signup` → `create_project` → `create_task` → `invite_member` → `teammate_joined` → `comment` → `upgrade`

## Every path is unique

In [ ]:
#| echo: false
from toy import (
    branching_figure,
    distinct_colors,
    first_visits_figure,
    stage_graph,
    taskboard_colors,
    visit_paths_figure,
)

USERS = ["user_00335", "user_00003", "user_00048", "user_00034"]
# The trees: a random 1,000 of the 20,000 users is enough to see the point.
sample = taskboard.sample_paths(n=1000, random_state=7)
event_colors_31 = distinct_colors(taskboard.df["event"])
HTML(
    first_visits_figure(taskboard, USERS, colors=event_colors_31)
    + branching_figure(sample, 15, colors=event_colors_31, size=250)
)

## Even the transition graph is tangled

In [ ]:
#| eval: false
taskboard.transition_graph()

In [ ]:
#| echo: false
stage_graph(taskboard, view={"edgeFilter": {"mode": "range", "range": [0, 1]}})

## What kind of visit was it?

In [ ]:
#| echo: false
import json

# The names the next slides give the clusters (checked against the profiles).
VISIT_NAMES = {
    "cluster_0": "solo",
    "cluster_1": "teamwork",
    "cluster_2": "billing",
    "cluster_3": "onboarding",
    "cluster_4": "waiting",
    "cluster_5": "check_in",
    "cluster_6": "power",
    "cluster_7": "planning",
}
clusters = taskboard.cluster_analysis(
    features=[{"metric": "event_count_bulk"}],
    method_args={"n_clusters": 8},
    scaler="std",
    path_col="session_id",
)
clusters.cluster_renames = json.dumps(VISIT_NAMES)
clusters.slide_height = 600  # the whole slide below the title
clusters

- `solo`: tasks created and completed alone
- `teamwork`: comments, assignments, mentions
- `billing`: pricing and checkout
- `onboarding`: signup, the tour, a first project
- `waiting`: checks on an invite, resends it
- `check_in`: a glance: the board, notifications, a report
- `power`: connecting an integration, exports
- `planning`: the timeline, due dates, dependencies

In [ ]:
#| echo: false
#| output: false
visits = (
    taskboard
        .add_clusters(
            "visit", features=[{"metric": "event_count_bulk"}],
            method_args={"n_clusters": 8}, scaler="std",
            path_col="session_id",
        )
        .collapse_events(group_col="session_id", name={"col": "visit"})
        .rename_events({
            "cluster_0": "solo",
            "cluster_1": "teamwork",
            "cluster_2": "billing",
            "cluster_3": "onboarding",
            "cluster_4": "waiting",
            "cluster_5": "check_in",
            "cluster_6": "power",
            "cluster_7": "planning",
        })
)

## Eight visit types instead of 31 events

In [ ]:
#| echo: false
stage_graph(visits, view={"edgeFilter": {"mode": "range", "range": [0, 1]}})

- After `onboarding`: → `solo` 68%, → `check_in` 22%
- The adoption ladder: `solo` → `waiting` → `teamwork` → `power`: an invite, a wait, a team
- `planning` → `planning` 49%: some plan visit after visit
- `check_in` → `path_end` 42%: a quick check-in is often the last visit
- Before: the same paths, event by event

In [ ]:
#| echo: false
stage_graph(taskboard, view={"edgeFilter": {"mode": "range", "range": [0, 1]}})

## The same paths, visit by visit

In [ ]:
#| echo: false
HTML(
    visit_paths_figure(visits, USERS)
    + branching_figure(visits.sample_paths(n=1000, random_state=7), 5, colors=taskboard_colors(visits.df["event"]), size=250)
)

## One visit, one event

In [ ]:
#| eval: false
#| code-line-numbers: "|3-7|8|9-18"
visits = (
    taskboard
        .add_clusters(
            "visit", features=[{"metric": "event_count_bulk"}],
            method_args={"n_clusters": 8}, scaler="std",
            path_col="session_id",
        )
        .collapse_events(group_col="session_id", name={"col": "visit"})
        .rename_events({
            "cluster_0": "solo",
            "cluster_1": "teamwork",
            "cluster_2": "billing",
            "cluster_3": "onboarding",
            "cluster_4": "waiting",
            "cluster_5": "check_in",
            "cluster_6": "power",
            "cluster_7": "planning",
        })
)

## Three tools, one story

- `cluster_analysis`: find what kinds of visits there are
- `add_clusters` + `collapse_events`: turn each visit into one event
- `transition_graph`: see the journey across visits

Part 4 comes back to Taskboard with an A/B test.

# Part 2: Introduction

## UX research

| | Qualitative: why, how to fix | Quantitative: how many, how much |
|---|---|---|
| Behavioral: what people do | Moderated usability tests, Field studies, Diary studies | Event-log analysis ← we are here, A/B tests, Usability benchmarks |
| Attitudinal: what people say | Interviews, Focus groups, Open-ended feedback | Surveys, Card sorting, Feedback text mining |

<small>After C. Rohrer, "A Landscape of User Research Methods", Nielsen Norman Group</small>

## Event-log data

| user_id | session_id | event | timestamp | country | platform | n_session |
|---|---|---|---|---|---|---|
| user_0042 | sess_0042_1 | home | 2024-03-13 10:21:14 | US | desktop | 1 |
| user_0042 | sess_0042_1 | catalog | 2024-03-13 10:21:28 | US | desktop | 1 |
| user_0042 | sess_0042_1 | product_view | 2024-03-13 10:22:12 | US | desktop | 1 |
| user_0042 | sess_0042_2 | cart | 2024-03-15 19:02:40 | US | mobile | 2 |

- path identifiers: user_id, *session_id*
- event as a string: event
- event time: timestamp
- segments: *country*, *platform*
- miscellaneous: *n_session*

## Why event-log analysis is challenging

1. Number of event types
   - After cleanup and grouping: ~10
   - Small products: ~100
   - Medium products: a few hundred
   - Large products: 1,000+
2. Messy data
   - Same action, different names: `add_to_cart` (web), `AddToCart` (iOS), `cart_add` (Android)
   - Renamed without backfill: `checkout_start` until May, `begin_checkout` after
   - Raw URLs as events: `/product/18237`, `/product/90311?color=black`, … instead of `product_page`
3. Noisy patterns
   - A→B→C
   - A→X→B→C? (extra step)
   - A→B→B→C? (repeat)
   - B→A→C? (different order)
4. No "average" path
   - Users X (50%): A→B→C
   - Users Y (50%): D→B→E
   - Half of `B`s come from `A`, half end in `E`
   - Yet nobody went A→B→E.

## Retentioneering

### An open-source Python library built for Quant UXR addressing these problems. Let's see how it works, on toy data first.

# Part 3: Retentioneering tools

## Eventstream

- Eventstream: the core class – a data container and the API entry point
- A wrap of pandas.DataFrame
- Schema assigns column roles

In [ ]:
import pandas as pd

df = pd.read_csv("events.csv")
df

In [ ]:
#| code-line-numbers: "|2|3|4|5|6"
stream = rete.Eventstream(df, schema={
    "path_cols": ["user_id", "session_id"],
    "event_col": "event",
    "timestamp_col": "timestamp",
    "segment_cols": ["country", "platform"],
    "custom_cols": ["n_session"],
})

## Data accessor: Eventstream.df

In [ ]:
stream.df

- `event_type`: `raw` for your events; synthetic ones like `path_start` get their own type
- `index`: the event's position in its path
- `subindex`: orders event types that share a timestamp

## 3.1 Data processors

## Types of data processors

Data processor is a method that changes Eventstream and returns its modified version

- Data processors
  - Adding events
    - `add_events`
    - `add_start_end_events`
  - Renaming, editing & merging events
    - `rename_events`
    - `edit_events`
    - `collapse_events`
    - `urls_to_events`
  - Filtering, truncating & sampling
    - `drop_events`
    - `filter_events`
    - `filter_paths` 
    - `truncate_paths`
    - `sample_paths`
  - Sessions & lifecycle
    - `split_sessions`
    - `to_daily_states`
  - Segments & clustering
    - `add_segment`
    - `drop_segment`
    - `add_clusters`
    - `rename_segment_levels`

## Ten toy paths

In [ ]:
#| echo: false
from IPython.display import HTML
from toy import (
    edge_weights_figure,
    grouped_paths_figure,
    metrics_figure,
    pairs_figure,
    paths_figure,
    toy_events,
)

stream = rete.Eventstream(toy_events(), schema={"segment_cols": ["group"]})
HTML(grouped_paths_figure(stream, "group"))

## filter_events: drop events

In [ ]:
no_x = stream.filter_events(drop={"event": ["X"]})

In [ ]:
#| echo: false
HTML(paths_figure(stream, no_x))

## filter_events: any SQL condition

In [ ]:
no_b_before_e = stream.filter_events(sql="""
    SELECT * FROM eventstream
    QUALIFY event != 'B'
        OR LEAD(event) OVER (PARTITION BY user_id ORDER BY timestamp) IS DISTINCT FROM 'E'
""")

In [ ]:
#| echo: false
HTML(paths_figure(stream, no_b_before_e))

## Path metrics: one number per path

In [ ]:
metrics = stream.get_metrics([
    {"metric": "length"},
    {"metric": "has_event", "metric_args": {"event": "X"}},
    {"metric": "matches_pattern", "metric_args": {"pattern": "A->.*->B->.*->C"}},
])

In [ ]:
#| echo: false
HTML(metrics_figure(stream, metrics))

## Path patterns: regex over events

### Syntax

- `->`: then
- `.`: any one event
- `.*`: any run of events, including none
- `[A|D]`: one event, A or D
- `[^X]`: one event, anything but X
- `[^X]*`: a run of events with no X
- `path_start`, `path_end`: the path's own boundaries

### Examples

- `A->B`: A, then B right after it
- `A->.*->C`: A, then C at some point later
- `[A|D]->B`: A or D, then B
- `A->[^X]*->C`: A, then C, with no X in between
- `path_start->[^A|D]`: paths that begin with neither A nor D
- `C->path_end`: paths that end with C

Patterns aren't tied to a path's start or end: `A->.*->C` also matches B→A→C. To pin the start, write `path_start->A->.*->C`.

## filter_paths: keep whole paths

Keep paths that **go from A to C** and either **contain X** or **visit B twice or more**.

In [ ]:
#| code-line-numbers: "|2|4|5"
noisy = stream.filter_paths({"op": "and", "args": [
    {"metric": "matches_pattern", "op": "=", "value": True, "metric_args": {"pattern": "A->.*->C"}},
    {"op": "or", "args": [
        {"metric": "has_event", "op": "=", "value": True, "metric_args": {"event": "X"}},
        {"metric": "event_count", "op": ">", "value": 1, "metric_args": {"event": "B"}},
    ]},
]})

In [ ]:
#| echo: false
HTML(paths_figure(stream, noisy, strike=True))

## truncate_paths: cut a window

In [ ]:
b_to_c = stream.truncate_paths(start_anchor="B", end_anchor="C")

In [ ]:
#| echo: false
HTML(paths_figure(stream, b_to_c, strike=True))

## Anchors: which B, and how far

In [ ]:
last_b = {"pattern": "B", "occurrence": "last"}
after_last_b = stream.truncate_paths(
    start_anchor=last_b,
    end_anchor={**last_b, "offset": 1},
)

In [ ]:
#| echo: false
HTML(paths_figure(stream, after_last_b, strike=True))

## Anchors: a list means whichever comes first

In [ ]:
b_to_c_or_end = stream.truncate_paths(start_anchor="B", end_anchor=["C", "path_end"])

In [ ]:
#| echo: false
HTML(paths_figure(stream, b_to_c_or_end, strike=True))

## Chaining data processors

In [ ]:
#| code-line-numbers: "|3|4|5"
clean = (
    stream
        .rename_events({"E": "C"})
        .drop_events(["X"])
        .filter_paths({"metric": "length", "op": ">=", "value": 3})
)

In [ ]:
#| echo: false
from toy import chain_steps_figure

HTML(chain_steps_figure(stream, clean))

## 3.2 Widgets: Funnel vs. Transition Graph

## Funnels: the traditional view

In [ ]:
#| output: false
stream.funnel(steps=["A", "B", "C"])

In [ ]:
#| echo: false
from toy import funnel_figure

HTML(funnel_figure(stream, ["A", "B", "C"]))

What a funnel can't see:

- Other routes: the 30% outside the funnel are a blank; their D→B→E is invisible
- Cycles: A→B→A→B→C counts the same as A→B→C
- Loops: A→B→B→C repeats B; the B→B step is invisible
- Detours: the X in A→X→B→C is invisible
- Other orders: B→A→C stops at A; the B before it is ignored

## Transition graph: count adjacent pairs

Every pair of neighboring events in every path, counted:

In [ ]:
#| echo: false
HTML(pairs_figure(stream))

In [ ]:
#| echo: false
stream.transition_graph(
    edge_weight="count",
    sidebar_open=False,
    view={"edgeFilter": {"mode": "range", "range": [0, 1]}},
)

## Edge weights

In [ ]:
#| echo: false
HTML(edge_weights_figure(
    stream,
    edges=[("A", "B"), ("B", "C"), ("B", "B"), ("B", "E")],
    weights=["count", "unique_paths", "proba_out", "proba_in"],
))

- `count`: how many times the transition happened
- `unique_paths`: how many paths have it; A→B→A→B→C counts once
- `proba_out`: its share of everything that follows the source
- `proba_in`: its share of everything that leads into the target
- Also: `share_of_total`, `avg_per_path`, `time_median`, `time_q95`

## Interactive features (live demo)

- Views
- Search event
- Edge filter
- Node focus + 2-3 clicks
- Selecting traces
- Ego view
- user/session path selector

In [ ]:
#| output: false
ecom = rete.datasets.load_ecom()
checkout = ["cart", "shipping_details", "payment_details", "purchase"]
ecom.transition_graph(views=[
    {"name": "Checkout", "focus": {"type": "path", "nodes": checkout}},
    {"name": "Payment errors",
     "focus": {"type": "node", "event": "payment_error", "direction": "both"}},
])

## 3.3 Widgets: Tree vs. Step matrix 

## Step matrix: count events at each step

In [ ]:
#| echo: false
from toy import prefix_tree_svg, step_layout, step_matrix_table, step_paths_table

HTML(step_layout(
    step_paths_table(stream, 6, spotlight=(1, 2, 3)),
    step_matrix_table(stream, 6, spotlight=(1, 2, 3)),
))

## Tree vs. step matrix

In [ ]:
#| echo: false
HTML(step_layout(
    prefix_tree_svg(stream, 6, spotlight=(1, 2, 3)),
    step_matrix_table(stream, 6, spotlight=(1, 2, 3)),
))

## Step matrix centered on an event

In [ ]:
#| echo: false
HTML(step_layout(
    step_paths_table(stream, 3, spotlight=(0, -1, 1), pattern="B"),
    step_matrix_table(stream, 3, spotlight=(0, -1, 1), pattern="B"),
))

## Why not tree-based diagrams?

In [ ]:
#| echo: false
from toy import branching_figure

HTML(branching_figure(ecom, 5))

> A tree branches on every distinct beginning. On real data, almost every path gets a branch of its own.

## Step matrix: one row per event (live demo)

The same 600 users: one row per event type, however many paths. Rows you don't need can be hidden.

In [ ]:
ecom.step_matrix()

## Step Sankey: the same numbers as flows

In [ ]:
stream.step_sankey()

# Part 4: A grey A/B test

## The Timeline experiment

- **Pricing:** Free for up to two people; Team, per seat, adds more members, reports and automations
- **Feature:** Timeline, a Gantt-style view of a project. It has always been there as a second tab, and few open it
- **Test:** a new project opens on Timeline, with a prompt: *Plan with your team*
- **Hypothesis:** on a timeline every task wants a date and an assignee; an empty assignee slot (+) invites a teammate right there. Plan the work, and you bring in the people to do it
- **Primary metric:** team activation, the share of new users whose teammate joins (`teammate_joined`): expected 27% → 30%
- **Design:** every new user over six weeks, 50/50: 20,000 users

In [ ]:
#| echo: false
from toy import timeline_mockups

HTML(timeline_mockups())

## The result: grey

In [ ]:
#| echo: false
from toy import ab_results_table

HTML(ab_results_table(taskboard, {
    "create_project": "Created a project",
    "invite_member": "Invited a teammate",
    "teammate_joined": "Team activation",
    "upgrade": "Upgrade",
}, primary="teammate_joined"))

> Nothing moved. Ship it, or roll it back?

## The funnel: grey at every step

In [ ]:
taskboard.funnel(
    steps=["signup", "create_project", "invite_member", "teammate_joined", "upgrade"],
    diff=("ab_group", "test", "control"),
)

<small>A funnel counts a user at a step only after every step before it, in order. Invited teammates (no `signup`) and users who pay without a team never reach its last step: 8% upgrade here, 13% of all users in the table before.</small>

## Through the funnel, and back

In [ ]:
#| echo: false
stage_graph(visits, view={"edgeFilter": {"mode": "range", "range": [0, 1]}})

In [ ]:
#| echo: false
back_to_solo = visits.step_matrix(path_pattern="teamwork", max_steps=4)
back_to_solo.slide_height = 250
back_to_solo

- `teamwork` → `solo`: one team visit in ten is followed by working alone again
- Step 0 is a user's first visit with the team. Two visits later, 26% work alone again; a quarter three visits later
- The funnel counted every one of them at `teammate_joined`, as activated

## Who is Timeline for?

In [ ]:
#| echo: false
USER_NAMES = {
    "cluster_0": "teams_paid",
    "cluster_1": "solo_users",
    "cluster_2": "invited",
    "cluster_3": "planners",
    "cluster_4": "payment_errors",
    "cluster_5": "dropped_off",
    "cluster_6": "teams_unpaid",
}
users = taskboard.cluster_analysis(
    features=[{"metric": "event_count_bulk"}],
    method_args={"n_clusters": 7},
    scaler="std",
    path_col="user_id",
    overview_metrics=[
        {"metric": "in_segment_bulk", "metric_args": {"segment_name": "ab_group"}},
    ],
)
users.cluster_renames = json.dumps(USER_NAMES)
users.slide_height = 260  # a short table: two rows per group
users

- `teams_paid`: a team that stuck together, and pays
- `solo_users`: work alone; most invited someone, for some a teammate came and went
- `invited`: joined through a teammate's invite
- `planners`: plan on the timeline, visit after visit
- `payment_errors`: hit the broken checkout; most wrote to support
- `dropped_off`: left during onboarding or after a few visits
- `teams_unpaid`: a teammate joined, no upgrade; many went back to working alone

## Timeline works, for planners

| | `solo_users` | `planners` |
|---|---|---|
| Who they are | want a team: invite, wait | freelancers, consultants: plan client work alone |
| Invited a teammate | 90% | 10% |
| Planning, of all visits | 0% | 43% |
| Upgrade | 1.8% | 4% |
| Test users | 51% | 80% |

- Timeline didn't bring teams in: planners don't invite, so team activation couldn't move
- It kept planners: in control most of them never find it, and drop off
- Team sells seats, and planners have no one to seat. Next: show Timeline to those who plan, and test a solo Pro plan for them

## Digging deeper: a loop on the timeline

In [ ]:
#| echo: false
planners = (
    taskboard
        .add_clusters(
            "user_type", features=[{"metric": "event_count_bulk"}],
            method_args={"n_clusters": 7}, scaler="std", path_col="user_id",
        )
        .filter_paths({"op": "=", "metric": "in_segment", "value": 1,
                       "metric_args": {"segment_name": "user_type", "segment_level": "cluster_3"}})
)
stage_graph(planners, view={"edgeFilter": {"mode": "range", "range": [0, 1]}})

- `view_timeline` → `view_timeline` 31%: the strongest edge out of the timeline. Planners open it, and open it again, in the same visit
- 41% of planners have this loop. They make 8 visits instead of 11, stay 16 days instead of 23, and pay 1% instead of 6%
- What are they doing there? The log can't say: nothing they do on that screen is tracked

## Digging deeper: what they click

- **Ask:** UX interviews and usability sessions with planners, at work on the timeline
- **Track:** every click on the timeline screen, with its coordinates
- **Find:** the task bars look clickable, but aren't. Users click, nothing happens, they open the timeline again
- **Fix:** a bar opens its task; drag it to reschedule

<small>A real case from Maxim Godzi's practice, replayed on Taskboard: users dropped off over UI elements they took for buttons. A loop on the transition graph gave it away; click coordinates showed where.</small>

In [ ]:
#| echo: false
from toy import timeline_clicks_mockups

HTML(timeline_clicks_mockups())